In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('flag_dominant_colours.csv')

In [3]:
df

,Country,Color1,Color2,Color3,Color4,Color5,Color6
0,Andorra,"(np.uint8(0), np.uint8(24), np.uint8(168))","(np.uint8(254), np.uint8(223), np.uint8(0))","(np.uint8(208), np.uint8(16), np.uint8(58))",NaN,NaN,NaN
1,United Arab Emirates,"(np.uint8(255), np.uint8(0), np.uint8(0))","(np.uint8(0), np.uint8(115), np.uint8(47))","(np.uint8(255), np.uint8(255), np.uint8(255))","(np.uint8(0), np.uint8(0), np.uint8(1))",NaN,NaN
2,Afghanistan,"(np.uint8(0), np.uint8(0), np.uint8(1))","(np.uint8(191), np.uint8(0), np.uint8(0))","(np.uint8(0), np.uint8(153), np.uint8(0))",NaN,NaN,NaN
3,Antigua and Barbuda,"(np.uint8(0), np.uint8(0), np.uint8(1))","(np.uint8(206), np.uint8(17), np.uint8(38))","(np.uint8(252), np.uint8(209), np.uint8(22))","(np.uint8(0), np.uint8(114), np.uint8(198))","(np.uint8(255), np.uint8(255), np.uint8(255))",NaN
4,Albania,"(np.uint8(255), np.uint8(0), np.uint8(0))","(np.uint8(0), np.uint8(0), np.uint8(1))",NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
192,Kosovo,"(np.uint8(36), np.uint8(74), np.uint8(165))","(np.uint8(208), np.uint8(166), np.uint8(80))",NaN,NaN,NaN,NaN
193,Yemen,"(np.uint8(241), np.uint8(6), np.uint8(0))","(np.uint8(255), np.uint8(255), np.uint8(255))","(np.uint8(0), np.uint8(0), np.uint8(1))",NaN,NaN,NaN
194,South Africa,"(np.uint8(0), np.uint8(120), np.uint8(71))","(np.uint8(255), np.uint8(255), np.uint8(255))","(np.uint8(225), np.uint8(57), np.uint8(45))","(np.uint8(255), np.uint8(185), np.uint8(21))","(np.uint8(0), np.uint8(0), np.uint8(1))","(np.uint8(0), np.uint8(12), np.uint8(138))"
195,Zambia,"(np.uint8(25), np.uint8(138), np.uint8(0))","(np.uint8(239), np.uint8(125), np.uint8(0))","(np.uint8(222), np.uint8(32), np.uint8(16))","(np.uint8(0), np.uint8(0), np.uint8(1))",NaN,NaN


In [6]:
#create a dataframe with colnames as country names and index as country names
countries = df['Country'].tolist()
distance_df = pd.DataFrame(index=countries, columns=countries)

In [26]:
import numpy as np

def rgb_distance(color1, color2):
    """
    Calculate Euclidean distance between two RGB colors.
    
    Parameters:
    - color1, color2: tuples of (r, g, b) values (can be np.uint8 or regular ints)
    
    Returns:
    - Euclidean distance in RGB space
    """
    # Handle the case where colors are tuples of np.uint8
    if len(color1) == 3:
        r1, g1, b1 = int(color1[0]), int(color1[1]), int(color1[2])
        r2, g2, b2 = int(color2[0]), int(color2[1]), int(color2[2])
    else:
        # If it's a different format, try to extract the RGB values
        r1, g1, b1 = color1
        r2, g2, b2 = color2
    
    return np.sqrt((r1 - r2)**2 + (g1 - g2)**2 + (b1 - b2)**2)

def parse_color_string(color_str):
    """
    Parse a color string like "(np.uint8(0), np.uint8(24), np.uint8(168))"
    into a tuple of integers (0, 24, 168).
    
    Parameters:
    - color_str: String representation of color tuple
    
    Returns:
    - Tuple of (r, g, b) as integers
    """
    import re
    # Extract all numbers from the string
    numbers = re.findall(r'\d+', color_str)
    # Convert to integers and return as tuple
    return tuple(int(n) for n in numbers[:3])

In [27]:
def extract_colors(country, df):
    """
    Extract all non-NaN colors for a given country.
    
    Parameters:
    - country: Country name
    - df: DataFrame with flag colors
    
    Returns:
    - List of RGB tuples (as integers)
    """
    row = df[df['Country'] == country].iloc[0]
    colors = []
    
    for col in ['Color1', 'Color2', 'Color3', 'Color4', 'Color5', 'Color6']:
        if pd.notna(row[col]):
            colors.append(parse_color_string(row[col]))
    
    return colors

In [28]:
def calculate_flag_distance(country1, country2, df, color_threshold=30):
    """
    Calculate distance between two flags based on color similarity.
    Uses Jaccard-style distance: 1 - (matched_colors / total_unique_colors)
    
    Parameters:
    - country1, country2: Country names
    - df: DataFrame with flag colors
    - color_threshold: Max RGB difference to consider colors "similar" (default: 30)
    
    Returns:
    - distance score (0 = identical colors, 1 = completely different)
    """
    # Step 1: Extract colors for both countries
    colors1 = extract_colors(country1, df)
    colors2 = extract_colors(country2, df)
    
    # Handle edge case where one or both flags have no colors
    if len(colors1) == 0 or len(colors2) == 0:
        return 1.0
    
    # Step 2: Find matches
    matched_count = 0
    
    for color1 in colors1:
        # Find the minimum distance to any color in flag2
        min_dist = min(rgb_distance(color1, color2) for color2 in colors2)
        
        # If within threshold, consider it a match
        if min_dist <= color_threshold:
            matched_count += 1
    
    # Step 3: Calculate Jaccard-style distance
    # Total unique colors = max(len(colors1), len(colors2))
    # This ensures that if a 3-color flag matches all colors of a 5-color flag,
    # the distance reflects that only 3 out of 5 possible colors matched
    total_colors = max(len(colors1), len(colors2))
    
    distance = 1 - (matched_count / total_colors)
    
    return distance

In [29]:
# To populate the entire distance matrix:
def populate_distance_matrix(df, distance_df, color_threshold=30):
    """
    Populate the distance matrix for all country pairs.
    
    Parameters:
    - df: DataFrame with flag colors
    - distance_df: Empty distance DataFrame to populate
    - color_threshold: Max RGB difference to consider colors "similar"
    
    Returns:
    - Populated distance DataFrame
    """
    countries = df['Country'].tolist()
    
    for i, country1 in enumerate(countries):
        for j, country2 in enumerate(countries):
            if i == j:
                # Distance to self is 0
                distance_df.loc[country1, country2] = 0.0
            else:
                # Calculate distance
                dist = calculate_flag_distance(country1, country2, df, color_threshold)
                distance_df.loc[country1, country2] = dist
        
        # Optional: print progress
        if (i + 1) % 20 == 0:
            print(f"Processed {i + 1}/{len(countries)} countries")
    
    return distance_df

In [30]:
distance_df = populate_distance_matrix(df, distance_df)

Processed 20/197 countries
Processed 40/197 countries
Processed 60/197 countries
Processed 80/197 countries
Processed 100/197 countries
Processed 120/197 countries
Processed 140/197 countries
Processed 160/197 countries
Processed 180/197 countries


In [31]:
distance_df

,Andorra,United Arab Emirates,Afghanistan,Antigua and Barbuda,Albania,Armenia,Angola,Argentina,Austria,Australia,...,Saint Vincent and the Grenadines,Venezuela,Vietnam,Vanuatu,Samoa,Kosovo,Yemen,South Africa,Zambia,Zimbabwe
Andorra,0.0,0.5,0.333333,0.4,0.333333,0.0,0.333333,0.666667,0.333333,0.333333,...,0.333333,0.0,0.333333,0.25,0.333333,0.666667,0.333333,0.5,0.25,0.4
United Arab Emirates,0.0,0.0,0.5,0.2,0.0,0.0,0.0,0.5,0.5,0.0,...,0.0,0.0,0.5,0.0,0.5,1.0,0.0,0.333333,0.0,0.2
Afghanistan,0.0,0.5,0.0,0.4,0.333333,0.0,0.333333,1.0,0.666667,0.333333,...,0.333333,0.0,0.666667,0.25,0.0,0.666667,0.333333,0.666667,0.5,0.4
Antigua and Barbuda,0.0,0.2,0.4,0.0,0.2,0.0,0.2,0.6,0.4,0.2,...,0.2,0.0,0.4,0.0,0.4,0.8,0.2,0.166667,0.0,0.0
Albania,0.333333,0.5,0.666667,0.6,0.0,0.333333,0.0,0.5,0.5,0.333333,...,0.333333,0.333333,0.5,0.5,0.5,1.0,0.333333,0.666667,0.5,0.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Kosovo,0.666667,1.0,0.666667,0.8,1.0,0.666667,1.0,1.0,0.5,1.0,...,0.666667,0.666667,0.5,0.75,0.5,0.0,1.0,0.833333,0.5,0.8
Yemen,0.0,0.25,0.666667,0.4,0.0,0.0,0.0,0.333333,0.333333,0.0,...,0.0,0.0,0.333333,0.25,0.666667,1.0,0.0,0.5,0.25,0.4
South Africa,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.5,0.5,0.0,...,0.0,0.0,0.5,0.0,0.333333,0.833333,0.0,0.0,0.0,0.0
Zambia,0.0,0.25,0.5,0.2,0.25,0.0,0.25,0.75,0.5,0.25,...,0.0,0.0,0.5,0.0,0.25,0.5,0.0,0.333333,0.0,0.2
